In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def detect_and_replace_outliers(df):
    """
    使用 IQR 方法检测并替换异常值
    异常值会被邻近的正常值替换
    """
    # 遍历每一列
    for col in df.columns:
        if df[col].dtype in ['int64', 'float64']:  # 对数值列进行处理
            # 计算四分位数
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            # 计算异常值的上下边界
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            # 标记异常值
            outliers = (df[col] < lower_bound) | (df[col] > upper_bound)

            # 替换异常值为相邻的正常值的平均值
            for i in range(1, len(df) - 1):
                if outliers[i]:
                    # 使用前一个和后一个值的平均值来替换异常值
                    df[col].iloc[i] = (df[col].iloc[i - 1] + df[col].iloc[i + 1]) / 2
    return df

# 获取当前工作目录
current_directory = os.getcwd()

# 获取当前目录下所有Excel文件
excel_files = [f for f in os.listdir(current_directory) if f.endswith('.xlsx') or f.endswith('.xls')]

# 跳过的文件名
skip_file = '2023_24_Shenzhen.xlsx'  # 假设是一个.xlsx文件，确保扩展名正确

# 遍历每个文件并填补空缺值，添加进度条
for file in tqdm(excel_files, desc="Processing Excel Files"):
    if file == skip_file:  # 如果文件是要跳过的文件，继续下一个文件
        print(f"Skipping file {file}")
        continue
    try:
        # 读取Excel文件
        df = pd.read_excel(file)

        # 查找空缺值并填补
        df_filled = df.fillna(method='ffill').fillna(method='bfill')  # 先用前值填充，再用后值填充

        # 异常值检测和替换
        df_filled = detect_and_replace_outliers(df_filled)

        # 保存到原文件中，覆盖原始数据
        df_filled.to_excel(file, index=False)
        print(f"Missing values filled and outliers replaced, file saved as {file}")

    except Exception as e:
        print(f"Error processing file {file}: {e}")
